# Notebook 5: Parallelisation with LangChain

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what **Parallelisation** is and when to use it.
2. Use `RunnableParallel` (and its dictionary shorthand) in LangChain.
3. Understand the difference between **sequential** and **parallel** execution and their trade-offs.
4. Use **Pydantic** with `JsonOutputParser` for structured parallel outputs.
5. Run a working **Financial News Multi-Analyzer** that processes news in parallel.

---

## Use Case: Real-Time Financial News Analyzer

> **Scenario**: A trading desk receives breaking financial news and needs multiple insights instantly — sentiment, entity extraction, and impact summary. Running these sequentially wastes precious seconds. Parallelisation delivers all three simultaneously.

---

## Sequential vs. Parallel Execution

```
SEQUENTIAL (slow):
─────────────────────────────────────────────────────────
Article → [Sentiment: 2s] → [Entities: 2s] → [Summary: 2s]
Total time: ~6 seconds
─────────────────────────────────────────────────────────

PARALLEL (fast):
─────────────────────────────────────────────────────────
                    ┌→ [Sentiment: 2s] ─┐
Article ────────────┼→ [Entities: 2s]  ─┼→ Combined Results
                    └→ [Summary: 2s]  ──┘
Total time: ~2 seconds (bottleneck = slowest single task)
─────────────────────────────────────────────────────────
```

---

## Key Concepts

| Concept | Description |
|---|---|
| **RunnableParallel** | Runs multiple chains simultaneously on the same input |
| **Dict shorthand** | `{"a": chain_a, "b": chain_b}` is equivalent to `RunnableParallel(a=chain_a, b=chain_b)` |
| **Independent tasks** | Parallelisation only works when tasks don't depend on each other's output |
| **Bottleneck** | Total time = time of the slowest parallel task |
| **JsonOutputParser** | Parses LLM output as structured JSON using a Pydantic schema |

## ⚙️ Step 1: Setup and Azure OpenAI Configuration

In [1]:
# Install required packages (uncomment if needed)
# !pip install langchain-openai langchain pydantic

import os
import time
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# ── Load credentials from .env ─────────────────────────────────────────────────
load_dotenv()

llm = AzureChatOpenAI(azure_deployment="gpt-4.1", temperature=0.1)
print("Azure OpenAI model initialized!")

Azure OpenAI model initialized!


## 📰 Step 2: The Sample News Articles

We'll test with two different news articles to demonstrate the system's versatility.

In [2]:
news_articles = {
    "nvidia": """
    Nvidia Corp (NVDA) shares surged 8% in after-hours trading following a massive earnings beat.
    The semiconductor giant reported Q4 revenue of $22.1 billion, driven by insatiable demand for 
    its H100 AI chips. CEO Jensen Huang stated that 'accelerated computing and generative AI have 
    hit the tipping point.' This news sent ripples through the entire technology sector, lifting 
    peers like AMD (AMD) and TSMC (TSM). Analysts at Goldman Sachs raised their price target 
    from $600 to $800.
    """,
    
    "fed_rate": """
    The Federal Reserve announced a surprise 50 basis point interest rate cut today, citing 
    cooling inflation data and growing concerns about labor market weakness. Fed Chair Jerome Powell 
    stated the decision was 'data-dependent' and signaled further cuts are possible in 2025.
    Treasury yields fell sharply, with the 10-year yield dropping to 3.8%. Bank stocks including 
    JPMorgan Chase (JPM) and Bank of America (BAC) fell 3-4%, while growth stocks in the 
    Nasdaq surged on the news.
    """
}

print("✅ Two sample news articles loaded:")
for key in news_articles:
    print(f"  • {key}: {len(news_articles[key])} characters")

✅ Two sample news articles loaded:
  • nvidia: 516 characters
  • fed_rate: 512 characters


## 🏗️ Step 3: Defining the Three Parallel Analysis Chains

Each chain analyzes the same article from a different angle.

In [3]:
# ── Chain 1: Market Impact Summary ──────────────────────────────────────────
summary_chain = (
    ChatPromptTemplate.from_template(
        "You are a financial journalist. Summarize the market impact of this news in exactly two sentences.\n\n"
        "Article: {article}"
    )
    | llm
    | StrOutputParser()
)

# ── Chain 2: Financial Entity Extraction (Structured JSON) ───────────────────
class FinancialEntities(BaseModel):
    companies: List[str] = Field(description="List of company names mentioned")
    tickers: List[str] = Field(description="List of stock ticker symbols (e.g., NVDA, AAPL)")
    sectors: List[str] = Field(description="List of industries or sectors mentioned")
    key_figures: List[str] = Field(description="Key people mentioned (e.g., CEO names, officials)")

entity_parser = JsonOutputParser(pydantic_object=FinancialEntities)
entity_chain = (
    ChatPromptTemplate.from_template(
        "Extract financial entities from this news article.\n\n"
        "{format_instructions}\n\n"
        "Article: {article}"
    ).partial(format_instructions=entity_parser.get_format_instructions())
    | llm
    | entity_parser
)

# ── Chain 3: Investor Sentiment ───────────────────────────────────────────────
sentiment_chain = (
    ChatPromptTemplate.from_template(
        """Analyze the investor sentiment of this financial news article.
        
        Article: {article}
        
        Provide:
        - Overall Sentiment: (Bullish / Bearish / Mixed / Neutral)
        - Confidence Level: (High / Medium / Low)
        - Primary Reason: (one sentence)
        """
    )
    | llm
    | StrOutputParser()
)

print("✅ Three parallel analysis chains defined:")
print("   1. Market Impact Summary (StrOutputParser)")
print("   2. Financial Entity Extraction (JsonOutputParser + Pydantic)")
print("   3. Investor Sentiment (StrOutputParser)")

✅ Three parallel analysis chains defined:
   1. Market Impact Summary (StrOutputParser)
   2. Financial Entity Extraction (JsonOutputParser + Pydantic)
   3. Investor Sentiment (StrOutputParser)


## 🔀 Step 4: Creating the Parallel Workflow

The dictionary shorthand `{"key": chain}` automatically creates a `RunnableParallel`. All chains receive the same input and run simultaneously.

In [4]:
# Method 1: Dictionary shorthand (most common)
parallel_analyzer = {
    "summary": summary_chain,
    "entities": entity_chain,
    "sentiment": sentiment_chain
}

# Method 2: Explicit RunnableParallel (equivalent)
# parallel_analyzer = RunnableParallel(
#     summary=summary_chain,
#     entities=entity_chain,
#     sentiment=sentiment_chain
# )

print("✅ Parallel workflow created.")
print("   All three chains will receive the same article and run simultaneously.")

✅ Parallel workflow created.
   All three chains will receive the same article and run simultaneously.


## 🚀 Step 5: Running the Parallel Analysis

In [5]:
def analyze_news(article_text: str, article_name: str = "Article"):
    """Run all three analyses in parallel and display results."""
    print(f"\n{'='*60}")
    print(f"Analyzing: {article_name}")
    print(f"{'='*60}")
    
    # Run in parallel using RunnableParallel
    parallel_run = RunnableParallel(
        summary=summary_chain,
        entities=entity_chain,
        sentiment=sentiment_chain
    )
    
    start = time.time()
    results = parallel_run.invoke({"article": article_text})
    elapsed = time.time() - start
    
    print(f"\n  Parallel execution time: {elapsed:.2f}s")
    print(f"\n  Market Impact Summary:")
    print(f"   {results['summary']}")
    print(f"\n  Financial Entities:")
    entities = results['entities']
    print(f"   Companies: {entities.get('companies', [])}")
    print(f"   Tickers:   {entities.get('tickers', [])}")
    print(f"   Sectors:   {entities.get('sectors', [])}")
    print(f"\n  Investor Sentiment:")
    for line in results['sentiment'].split('\n'):
        if line.strip():
            print(f"   {line.strip()}")
    
    return results, elapsed

# Test with Nvidia earnings news
nvidia_results, nvidia_time = analyze_news(news_articles["nvidia"], "Nvidia Earnings")


Analyzing: Nvidia Earnings

  Parallel execution time: 1.85s

  Market Impact Summary:
   Nvidia's blockbuster earnings and bullish outlook triggered an 8% after-hours surge in its shares, sparking a rally across the broader technology sector and boosting peers like AMD and TSMC. The results prompted Goldman Sachs to hike its price target on Nvidia from $600 to $800, underscoring heightened investor optimism around AI-driven growth.

  Financial Entities:
   Companies: ['Nvidia Corp', 'AMD', 'TSMC', 'Goldman Sachs']
   Tickers:   ['NVDA', 'AMD', 'TSM']
   Sectors:   ['semiconductor', 'technology']

  Investor Sentiment:
   - Overall Sentiment: Bullish
   - Confidence Level: High
   - Primary Reason: The article highlights Nvidia's strong earnings beat, surging share price, positive analyst upgrades, and sector-wide optimism driven by AI demand.


## 🧪 Step 6: Testing with Federal Reserve News

In [6]:
# Test with Fed rate cut news
fed_results, fed_time = analyze_news(news_articles["fed_rate"], "Fed Rate Cut")


Analyzing: Fed Rate Cut

  Parallel execution time: 2.47s

  Market Impact Summary:
   The surprise 50 basis point rate cut by the Federal Reserve sent Treasury yields tumbling, with the 10-year yield dropping to 3.8%. Bank stocks declined sharply while growth stocks, particularly in the Nasdaq, rallied as investors anticipated a more accommodative monetary policy.

  Financial Entities:
   Companies: ['Federal Reserve', 'JPMorgan Chase', 'Bank of America', 'Nasdaq']
   Tickers:   ['JPM', 'BAC']
   Sectors:   ['Banking', 'Finance', 'Growth stocks']

  Investor Sentiment:
   - Overall Sentiment: Mixed
   - Confidence Level: High
   - Primary Reason: While the rate cut and falling yields boosted growth stocks, the move was prompted by concerns over economic weakness, leading to declines in bank stocks and signaling both positive and negative investor reactions.


## 📈 Step 7: Sequential vs. Parallel Timing Comparison

In [7]:
# Run the same analysis sequentially for comparison
print("Running sequential analysis for comparison...")
start_seq = time.time()
_ = summary_chain.invoke({"article": news_articles["nvidia"]})
_ = entity_chain.invoke({"article": news_articles["nvidia"]})
_ = sentiment_chain.invoke({"article": news_articles["nvidia"]})
seq_time = time.time() - start_seq

print(f"\n⏱️  Sequential time: {seq_time:.2f}s")
print(f"⏱️  Parallel time:   {nvidia_time:.2f}s")
print(f"🚀 Speedup:          {seq_time/nvidia_time:.1f}x faster")

Running sequential analysis for comparison...

⏱️  Sequential time: 2.64s
⏱️  Parallel time:   1.85s
🚀 Speedup:          1.4x faster


## 📚 Summary and Key Takeaways

| Pattern | When to Use | Limitation |
|---|---|---|
| **Sequential** | Task B needs Task A's output | Slow for independent tasks |
| **Parallel** | Tasks are independent of each other | All tasks get the same input |
| **Orchestrator-Worker** | Tasks need different inputs | More complex setup (see Notebook 6) |

### 🔑 Key Insights

1. **Independence is the prerequisite**: Parallelisation only works when tasks don't depend on each other's output. If Sentiment needs the Entities first, you can't parallelize them.
2. **Bottleneck determines total time**: If one chain takes 5s and others take 2s, the total is ~5s, not 2s.
3. **Same input, multiple perspectives**: Parallelisation is ideal for "analyze this from multiple angles" use cases — news analysis, document review, risk assessment.
4. **Orchestrator-Worker is the dynamic version**: When tasks need different inputs or the task list isn't predetermined, use the Orchestrator-Worker pattern (Notebook 6).

### 🚀 Next Steps
- **Notebook 6**: Learn the Orchestrator-Worker pattern for dynamic, multi-input parallel workflows.
